# RAGLedger — reproducible Google Colab GPU workflow

This notebook is the executable Colab companion to `docs/COLAB_RUNBOOK.md`. It is
one linear workflow: GPU check → pinned clone → Drive → install → model caches →
dataset subsets → functional preflight → **integration pilot** → optional
**separate research run** → resume → results export.

**Scope rule.** The pilot uses Qwen2.5-7B via Ollama q4_K_M as an explicitly
different model from the research configuration. It proves integration only.
The research configuration remains Contriever + DeBERTa NLI + Llama-3.1-8B-Instruct
(unquantised bfloat16, vLLM). The two are never equivalent and their outputs are
stored in different directories. Answer-level ASR remains **NOT MEASURED**.

Before running: **Runtime → Change runtime type → GPU**. The notebook does not
download the full NQ corpus or a large generator without an explicit execution
flag. It was not run in Colab during this repository update; see the static
validation report at the end.


## 1. Runtime selection and GPU verification

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, time, urllib.request

def run(cmd):
    return subprocess.run(cmd, text=True, capture_output=True, check=False)

gpu = run(["nvidia-smi", "-L"])
print(gpu.stdout or gpu.stderr)
GPU_PRESENT = gpu.returncode == 0 and "GPU" in gpu.stdout
print(json.dumps({"gpu_present": GPU_PRESENT, "gpu": gpu.stdout.strip(),
                  "python": sys.version.split()[0], "cpu_count": os.cpu_count(),
                  "disk_free_gb": round(shutil.disk_usage("/content").free / 1e9, 2)}, indent=2))
if not GPU_PRESENT:
    raise RuntimeError("No CUDA GPU. Runtime -> Change runtime type -> GPU, then rerun this cell.")

## 2. Clone the correct repository and pin the commit

In [ ]:
REPO_URL = "https://github.com/athishss/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git"
PINNED_COMMIT = "e2f26599ac2705cab856a5b401285d18b160d046"
REPO = pathlib.Path("/content/RAGLedger")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", PINNED_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", PINNED_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == PINNED_COMMIT, (actual, PINNED_COMMIT)
print("repository:", REPO_URL)
print("checked-out SHA:", actual)
os.environ["RAGLEDGER_COMMIT"] = actual

## 3. Mount Google Drive; keep active state in /content

Active, temporary execution state lives in `/content/ragledger_active/runs/<run-id>/`:
the FAISS index, provenance/answer/trust SQLite databases, checkpoint and stream file.
**`/content` is deleted by a Colab runtime reset.** Google Drive separately stores
datasets, model/config locks, preflight logs, exports and atomic completed-step snapshots
under `MyDrive/RAGLedger/`; the prepared index snapshot and SQLite backups are the
persistent recovery source. This keeps SQLite/FAISS I/O off the Drive FUSE mount. The
runner computes the canonical run signature and restores only a matching Drive snapshot,
after that signature is known and before any resume attempt.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = pathlib.Path("/content/drive/MyDrive/RAGLedger")
ACTIVE = pathlib.Path("/content/ragledger_active")
ACTIVE_RUNS = ACTIVE / "runs"
BACKUPS = DRIVE / "backups"
for name in ("datasets/raw", "datasets/pilot", "datasets/research", "configs",
             "preflight", "exports", "logs", "backups"):
    (DRIVE / name).mkdir(parents=True, exist_ok=True)
PREFLIGHT_PARENT = ACTIVE / "preflight"
for path in (ACTIVE_RUNS, BACKUPS, PREFLIGHT_PARENT):
    path.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
pathlib.Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("persistent root:", DRIVE)
print("active run root:", ACTIVE_RUNS)
print("snapshot root:", BACKUPS)
print("HF cache (temporary):", os.environ["HF_HOME"])


## 4. Install dependencies and verify the environment

In [ ]:
CONSTRAINTS = REPO / "constraints/colab.txt"
# Install without importing torch first: Colab base images change, and a
# pre-install import/version assertion is a common source of notebook failure.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all,data]"], check=True)

import faiss, transformers, pyarrow, numpy, scipy, sklearn, pydantic, yaml
import matplotlib, torch, trace_rag
VERSIONS = {
    "torch": torch.__version__.split("+")[0], "transformers": transformers.__version__,
    "faiss": getattr(faiss, "__version__", "unknown"), "pyarrow": pyarrow.__version__,
    "numpy": numpy.__version__, "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__, "pydantic": pydantic.__version__,
    "PyYAML": yaml.__version__, "matplotlib": matplotlib.__version__,
}
print(json.dumps(VERSIONS, indent=2))
assert torch.cuda.is_available(), "PyTorch cannot see the GPU after installation"
probe = torch.randn(256, 256, device="cuda")
assert torch.isfinite((probe @ probe).sum()).item()
print("GPU smoke test:", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda)
print("trace_rag import:", trace_rag.__file__)
assert str(trace_rag.__file__).startswith(str(REPO)), "imported a stale trace_rag copy"
pinned = {}
for line in CONSTRAINTS.read_text(encoding="utf-8").splitlines():
    if "==" in line and not line.strip().startswith("#"):
        name, version = line.strip().split("==", 1)
        pinned[name] = version
drift = {name: {"required": version, "installed": VERSIONS.get(name)}
         for name, version in pinned.items()
         if name in VERSIONS and VERSIONS[name] != version}
if drift:
    print("WARNING: resolved versions differ from declared constraints:")
    print(json.dumps(drift, indent=2))
(DRIVE / "logs" / "environment.json").write_text(json.dumps({
    "versions": VERSIONS, "constraint_drift": drift,
    "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}, indent=2))
print("environment artifact written")

## 5. Cache the embedding and NLI weights (user-controlled)

Weights are pinned to exact Hugging Face commit SHAs before the functional
preflight enables offline mode. This means the doctor cannot silently download a
different revision. The generator for the pilot is pulled separately below.


In [ ]:
from huggingface_hub import snapshot_download

EMBED_ID = "facebook/contriever"
EMBED_REV = "2bd46a25019aeea091fd42d1f0fd4801675cf699"
NLI_ID = "cross-encoder/nli-deberta-v3-base"
NLI_REV = "6c749ce3425cd33b46d187e45b92bbf96ee12ec7"
LOCK = {"embedding": {"model": EMBED_ID, "revision": EMBED_REV},
        "nli": {"model": NLI_ID, "revision": NLI_REV}}
for item in LOCK.values():
    path = snapshot_download(item["model"], revision=item["revision"])
    print(item["model"], item["revision"], "->", path)
(DRIVE / "configs" / "model_pins.json").write_text(json.dumps(LOCK, indent=2))


## 6. Pull the explicitly alternate pilot generator

This is **Qwen2.5-7B via Ollama q4_K_M**, not the research Llama model. The
notebook records the pulled manifest digest and actual quantisation, then passes
those exact values to both preflight and runner.


In [ ]:
OLLAMA_VERSION = "0.5.7"
if not shutil.which("zstd"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
if not shutil.which("ollama"):
    archive = pathlib.Path("/tmp/ollama-linux-amd64.tgz")
    subprocess.run(["curl", "-fL", f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/ollama-linux-amd64.tgz", "-o", str(archive)], check=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", "/usr/local"], check=True)
observed = subprocess.check_output(["ollama", "--version"], text=True).strip()
assert OLLAMA_VERSION in observed, (observed, OLLAMA_VERSION)
print("Ollama runtime:", observed)
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        return True
    except Exception:
        return False
if not ollama_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "ab"), stderr=subprocess.STDOUT)
for _ in range(30):
    if ollama_up():
        break
    time.sleep(2)
assert ollama_up(), "Ollama API did not start; inspect /tmp/ollama.log"
PILOT_MODEL = "qwen2.5:7b"
subprocess.run(["ollama", "pull", PILOT_MODEL], check=True)
tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))
entry = next(m for m in tags["models"] if m["name"] == PILOT_MODEL)
QWEN_DIGEST = entry["digest"]
request = urllib.request.Request("http://127.0.0.1:11434/api/show",
    data=json.dumps({"name": PILOT_MODEL}).encode(), headers={"Content-Type": "application/json"})
shown = json.load(urllib.request.urlopen(request))
QWEN_QUANT = shown.get("details", {}).get("quantization_level", "UNSPECIFIED")
assert QWEN_DIGEST.startswith("sha256:"), QWEN_DIGEST
pilot_lock = {"tag": PILOT_MODEL, "digest": QWEN_DIGEST, "quantization": QWEN_QUANT,
              "serving_version": OLLAMA_VERSION}
(DRIVE / "configs" / "pilot_model_lock.json").write_text(json.dumps(pilot_lock, indent=2))
print("pilot generator lock:", pilot_lock)
assert QWEN_QUANT != "UNSPECIFIED"


## 7. Download raw NQ data and build separate pilot/research subsets

The full raw BEIR file is downloaded once. Gold passages are always retained;
the fill is deterministic. The pilot gets 5,000 passages / 20 queries. The
research preparation is gated and gets 200,000 / 500.


In [ ]:
# The raw BEIR NQ corpus contains ~2.68M passages and occupies several GB.
# Downloading it is deliberately opt-in: do not let a "Run all" silently consume
# Drive bandwidth and storage. Set this to True after checking free space, or
# point RAW at an existing corpus.parquet/queries.parquet/qrels.tsv.
RUN_DATASET_DOWNLOAD = False
RUN_RESEARCH = False
RAW = DRIVE / "datasets" / "raw"
REQUIRED_RAW = ("corpus.parquet", "queries.parquet", "qrels.tsv")
missing_raw = [name for name in REQUIRED_RAW if not (RAW / name).exists()]
if missing_raw:
    if not RUN_DATASET_DOWNLOAD:
        raise RuntimeError(
            "Raw BEIR NQ data is missing: " + ", ".join(missing_raw) + ". "
            "This notebook does not download the multi-gigabyte NQ corpus implicitly. "
            "Set RUN_DATASET_DOWNLOAD=True in this cell after checking Google Drive "
            "space, or copy existing raw files into " + str(RAW) + ".")
    print("Downloading the raw BEIR NQ files to", RAW,
          "(several GB; this is the explicit opt-in requested above)")
    subprocess.run([sys.executable, str(REPO / "scripts/download_data.py"), "--dataset", "nq",
                    "--out", str(RAW), "--queries", "500"], cwd=REPO, check=True)
assert all((RAW / name).exists() for name in REQUIRED_RAW), (
    "Raw NQ files incomplete; inspect the download log and rerun this cell.")
print("raw corpus bytes:", (RAW / "corpus.parquet").stat().st_size)


In [ ]:
import importlib.util
from trace_rag.ingestion.subset import select_query_ids

spec = importlib.util.spec_from_file_location("download_data", REPO / "scripts/download_data.py")
dd = importlib.util.module_from_spec(spec); spec.loader.exec_module(dd)
gold = dd.read_qrels(RAW / "qrels.tsv")

def prepare(outdir, passages, queries, seed=20260921):
    outdir.mkdir(parents=True, exist_ok=True)
    chosen = select_query_ids(gold, queries, seed)
    wanted = set().union(*(gold[q] for q in chosen))
    corpus = outdir / "corpus_subset.parquet"
    qfile = outdir / "queries_subset.jsonl"
    qrels = outdir / "qrels.tsv"
    dd.build_subset(RAW / "corpus.parquet", wanted, passages, corpus, seed)
    dd.write_queries_subset(RAW / "queries.parquet", chosen, qfile)
    shutil.copy2(RAW / "qrels.tsv", qrels)
    contract = dd.verify_subset(corpus, qfile, qrels, passages, queries)
    if not contract["ok"]:
        raise RuntimeError("subset contract failed: " + "; ".join(contract["errors"]))
    print("subset contract:", json.dumps(contract, indent=2))
    assert contract["corpus_rows"] == passages
    assert contract["query_rows"] == queries
    assert contract["gold_coverage"] == 1.0
    return corpus, qfile, qrels, contract

PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS, PILOT_CONTRACT = prepare(
    DRIVE / "datasets/pilot", 5000, 20)
print("pilot ready:", PILOT_CORPUS)
if RUN_RESEARCH:
    RESEARCH_CORPUS, RESEARCH_QUERIES, RESEARCH_QRELS, RESEARCH_CONTRACT = prepare(
        DRIVE / "datasets/research", 200000, 500)
    print("research ready:", RESEARCH_CORPUS)
else:
    print("research data not prepared (RUN_RESEARCH=False)")


### Build evaluator-only gold answers for real ASR

NQ Open supplies short answers for the NQ questions. Matching is by normalized
question text; coverage must be 100% for the selected pilot queries. These
answers are passed only to the evaluator, never to the retriever, verifier or
generator. Without this file, answer-level ASR remains `NOT MEASURED`.

In [ ]:
PILOT_ANSWERS = DRIVE / "datasets/pilot/answers.jsonl"
cmd = [sys.executable, str(REPO / "scripts/prepare_nq_answers.py"),
       "--queries", str(PILOT_QUERIES), "--out", str(PILOT_ANSWERS)]
print(" ".join(cmd))
proc = subprocess.run(cmd, cwd=REPO, text=True)
if proc.returncode != 0:
    raise RuntimeError("NQ gold-answer coverage is incomplete; refusing to run an ASR experiment")
assert PILOT_ANSWERS.exists()
print("gold answers:", PILOT_ANSWERS)

## 8. Functional preflight for the pilot configuration

The same config and **same `--set` overrides** are passed here and to the runner.
Preflight runs functional embedding, generation and NLI inference with offline
HF flags, checks the endpoint and persistent SQLite ledger, and must return
`READY` / `PROTECTED`.


In [ ]:
PILOT_CONFIG = REPO / "config/colab_pilot_qwen.yaml"
PILOT_RUN = ACTIVE_RUNS / "pilot_qwen25_7b_q4"
PILOT_BACKUP = BACKUPS / "pilot_qwen25_7b_q4"
PILOT_OVERRIDES = [f"embedding.revision={EMBED_REV}", f"security.nli_revision={NLI_REV}",
                   f"generation.revision={QWEN_DIGEST}", f"generation.quantization={QWEN_QUANT}",
                   f"generation.serving_version={OLLAMA_VERSION}", f"storage.root={PILOT_RUN}"]

def preflight(config, corpus, queries, qrels, overrides, label, expected_storage_root):
    fresh = not expected_storage_root.exists()
    cmd = [sys.executable, str(REPO / "scripts/check_research_environment.py"),
           "--config", str(config), "--functional", "--json",
           "--preflight-storage-root", str(PREFLIGHT_PARENT / label),
           "--corpus", str(corpus), "--queries", str(queries), "--qrels", str(qrels)]
    for item in overrides:
        cmd += ["--set", item]
    proc = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    (DRIVE / "preflight" / f"{label}.log").write_text(proc.stdout + proc.stderr)
    payload = json.loads(proc.stdout[proc.stdout.find("{"):])
    (DRIVE / "preflight" / f"{label}.json").write_text(json.dumps(payload, indent=2))
    isolation = (payload.get("functional_doctor") or {}).get("storage_isolation") or {}
    assert isolation.get("schema") == "ragledger.preflight.storage.v1", isolation
    assert pathlib.Path(isolation["intended_root"]).resolve() == expected_storage_root.resolve(), isolation
    assert pathlib.Path(isolation["probe_root"]).resolve() != expected_storage_root.resolve(), isolation
    assert isolation.get("cleaned") is True, isolation
    assert not pathlib.Path(isolation["probe_root"]).exists(), isolation
    if fresh:
        assert not expected_storage_root.exists(), (
            "functional preflight contaminated a fresh experiment directory: "
            + str(expected_storage_root))
    print(label, payload["status"], payload["status_reason"])
    print("preflight storage isolated:", isolation["probe_root"], "-> cleaned")
    if payload["status"] != "READY":
        print(json.dumps(payload.get("failures", []), indent=2))
        raise RuntimeError(f"preflight is {payload['status']}, not READY")
    print("checked model:", payload["config"]["generation_model"], "| backend:",
          payload["config"]["generation_backend"])
    return payload

PILOT_PREFLIGHT = preflight(PILOT_CONFIG, PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS,
                            PILOT_OVERRIDES, "pilot", PILOT_RUN)
ps = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/ps"))
running = next((m for m in ps.get("models", []) if m.get("name") == PILOT_MODEL), None)
assert running, "Ollama model is not loaded"
assert running.get("digest") == QWEN_DIGEST, (running.get("digest"), QWEN_DIGEST)
assert int(running.get("size_vram", 0)) > 0, "Ollama model is not GPU-resident"
server_version = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/version"))["version"]
assert server_version == OLLAMA_VERSION, (server_version, OLLAMA_VERSION)
print("generator runtime verified:", QWEN_DIGEST, "Ollama", server_version,
      "size_vram =", running["size_vram"])


## 9. Integration pilot (5,000 passages, 20-query pool, 12 Zipf steps)

This is a component/integration pilot, not a research result. The 12 stream steps
sample with replacement; the manifest records the 20-query pool, 12 steps and the
actual number of distinct queries.


In [ ]:
def run_stream(config, corpus, queries, qrels, out, backup, purpose, steps, targets, limit, overrides):
    # The runner reuses a completed run only when its canonical signature matches.
    # The runner computes the canonical run signature and restores only a
    # Drive snapshot with that exact signature.  Do not duplicate signature
    # logic in the notebook.
    cmd = [sys.executable, str(REPO / "scripts/run_trust_stream.py"),
           "--config", str(config), "--corpus", str(corpus), "--queries", str(queries),
           "--qrels", str(qrels), "--out", str(out), "--backup-dir", str(backup),
           "--run-purpose", purpose, "--steps", str(steps), "--targets", str(targets),
           "--restore"]
    if limit is not None:
        cmd += ["--limit", str(limit)]
    for item in overrides:
        cmd += ["--set", item]
    if (out / "checkpoint.json").exists() and not (out / "metrics.json").exists():
        cmd.append("--resume")
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"stream failed with exit {proc.returncode}")
    return json.loads((out / "run_manifest.json").read_text())

PILOT_MANIFEST = run_stream(PILOT_CONFIG, PILOT_CORPUS, PILOT_QUERIES, PILOT_QRELS,
                            PILOT_RUN, PILOT_BACKUP, "pilot", 12, 2, 5000, PILOT_OVERRIDES)
assert PILOT_MANIFEST["run_purpose"] == "pilot"
assert PILOT_MANIFEST["real_model"] is True
assert PILOT_MANIFEST["models"]["equivalence_claim"] is False
assert PILOT_MANIFEST["models"]["generation"]["model_id"] == PILOT_MODEL
assert PILOT_MANIFEST["models"]["generation"]["revision"] == QWEN_DIGEST
assert PILOT_MANIFEST["models"]["generation"]["digest_matches_config"] is True
assert PILOT_MANIFEST["models"]["generation"]["runtime_version"] == OLLAMA_VERSION
assert PILOT_MANIFEST["query_pool_size"] == 20
assert PILOT_MANIFEST["stream_steps_completed"] == 12
assert PILOT_MANIFEST["models"]["verifier"]["backend"] == "nli"
assert PILOT_MANIFEST["dataset_contract"]["corpus_rows"] == 5000
print(json.dumps({k: PILOT_MANIFEST[k] for k in ("state", "run_purpose", "query_pool_size",
      "stream_steps_completed", "distinct_queries_evaluated", "research_findings_allowed")}, indent=2))


## 10. Separate research execution (gated)

Set `RUN_RESEARCH=True` only on a GPU that can serve the pinned Llama-3.1-8B
research model. The code installs/starts vLLM, runs the functional preflight on
the **same** research config and overrides, then evaluates 500 unique queries
without replacement. This still is **not** the full attack × baseline matrix and
answer-level ASR remains NOT MEASURED.

The vLLM weight download is user-controlled by this flag. A free T4 is not a
supported research host for this model.


In [ ]:
if not RUN_RESEARCH:
    print("Research run skipped: RUN_RESEARCH=False. This is intentional; prepare the 200K dataset and use an adequate GPU first.")
else:
    LLAMA_ID = "meta-llama/Llama-3.1-8B-Instruct"
    LLAMA_REV = "a09a35458c702b33eeacc393d103063234e8bc28"
    VLLM_VERSION = "0.7.3"
    RESEARCH_CONFIG = REPO / "config/nq_gpu.yaml"
    RESEARCH_RUN = ACTIVE_RUNS / "research_llama31_8b_bf16"
    RESEARCH_BACKUP = BACKUPS / "research_llama31_8b_bf16"
    RESEARCH_OVERRIDES = [f"embedding.revision={EMBED_REV}", f"security.nli_revision={NLI_REV}",
                          f"generation.revision={LLAMA_REV}", "generation.backend=openai",
                          f"generation.model_name={LLAMA_ID}", "generation.precision=bfloat16",
                          f"generation.serving_version={VLLM_VERSION}",
                          "generation.base_url=http://127.0.0.1:8000/v1",
                          f"storage.root={RESEARCH_RUN}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(CONSTRAINTS),
                    f"vllm=={VLLM_VERSION}"], check=True)
    vllm_log = open(DRIVE / "logs" / "vllm.log", "ab")
    subprocess.Popen(["vllm", "serve", LLAMA_ID, "--revision", LLAMA_REV,
                      "--port", "8000", "--dtype", "bfloat16", "--gpu-memory-utilization", "0.55"],
                     stdout=vllm_log, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2).read(); break
        except Exception: time.sleep(5)
    RESEARCH_PREFLIGHT = preflight(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                                   RESEARCH_QRELS, RESEARCH_OVERRIDES, "research", RESEARCH_RUN)
    RESEARCH_MANIFEST = run_stream(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                                   RESEARCH_QRELS, RESEARCH_RUN, RESEARCH_BACKUP, "research", 500, 5,
                                   None, RESEARCH_OVERRIDES)
    assert RESEARCH_MANIFEST["query_pool_size"] == 500
    assert RESEARCH_MANIFEST["stream_steps_completed"] == 500
    assert RESEARCH_MANIFEST["distinct_queries_evaluated"] == 500
    assert RESEARCH_MANIFEST["dataset_contract"]["corpus_rows"] == 200000
    assert RESEARCH_MANIFEST["dataset_contract"]["gold_coverage"] == 1.0
    assert RESEARCH_MANIFEST["research_findings_allowed"] is True
    assert RESEARCH_MANIFEST["research_evaluation"]["full_attack_baseline_matrix"] is False
    print("Research trust-stream run completed; this is not the full C6 benchmark.")


In [ ]:
# Optional separate Zipf trust-lifecycle experiment on the research corpus.
# It is NOT a 500-unique-query evaluation. It has its own active run and backup.
RUN_RESEARCH_ZIPF = False
if RUN_RESEARCH_ZIPF and RUN_RESEARCH:
    LIFE_RUN = ACTIVE_RUNS / "research_zipf_lifecycle"
    LIFE_BACKUP = BACKUPS / "research_zipf_lifecycle"
    LIFE_MANIFEST = run_stream(RESEARCH_CONFIG, RESEARCH_CORPUS, RESEARCH_QUERIES,
                               RESEARCH_QRELS, LIFE_RUN, LIFE_BACKUP, "research_lifecycle", 500, 5,
                               None, RESEARCH_OVERRIDES)
    print("Zipf lifecycle steps:", LIFE_MANIFEST["stream_steps_completed"],
          "distinct queries:", LIFE_MANIFEST["distinct_queries_evaluated"])
else:
    print("Zipf research-lifecycle run skipped (separate from the 500-unique-query research run).")


## 11. Resume after a runtime disconnection

Re-running the pilot cell resumes only when `checkpoint.json`, the saved index,
the SQLite ledger and the full run signature still match. Completed steps are
read from `stream.jsonl`. If the disconnect happened *inside* a step, resume is
refused rather than mixing partial trust effects. This cell only inspects state;
the run cell performs the safe resume.


In [ ]:
from trace_rag.snapshots import snapshot_summary

def inspect_run(path, backup):
    ckpt = path / "checkpoint.json"
    if ckpt.exists():
        c = json.loads(ckpt.read_text())
        print(path, "active phase=", c.get("phase"), "completed=", c.get("completed_steps"),
              "in_flight=", c.get("in_flight_step"))
    else:
        print(path, "has no active checkpoint")
    summary = snapshot_summary(backup)
    print(backup, "latest valid Drive snapshot:",
          summary.get("completed_steps") if summary else "none")

inspect_run(PILOT_RUN, PILOT_BACKUP)
if RUN_RESEARCH and (ACTIVE_RUNS / "research_llama31_8b_bf16").exists():
    inspect_run(ACTIVE_RUNS / "research_llama31_8b_bf16",
                BACKUPS / "research_llama31_8b_bf16")


## 12. Metrics, manifest and visualization

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

def show_results(path):
    m = json.loads((path / "metrics.json").read_text())
    manifest = json.loads((path / "run_manifest.json").read_text())
    print("run:", manifest["run_purpose"], "| state:", manifest["state"])
    print("models:", json.dumps(manifest["models"], indent=2))
    for key in ("steps", "query_pool_size", "query_sampling", "distinct_queries_evaluated",
                "poison_retrieved_rate", "poison_exposure_rate", "poison_citation_rate",
                "poison_quarantined", "clean_false_quarantine", "answer_level_asr"):
        if key in m: print(f"{key:>28}: {m[key]}")
    print("answer-level ASR:", m.get("answer_level_asr", "NOT MEASURED"))
    for png in sorted((path / "figures").glob("*.png"))[:4]:
        display(Image(filename=str(png)))
show_results(PILOT_RUN)


## 13. Train and freeze the suspicion scorer

The scorer is CPU logistic regression plus isotonic calibration. GPU work is
embedding, NLI and generation. Rows are captured at retrieval time; splitting is
by question, and the test split is never used for fit, calibration or thresholds.

In [ ]:
RUN_SCORER_TRAINING = True
SCORER_DIR = ACTIVE_RUNS / "scorer"; SCORER_DIR.mkdir(parents=True, exist_ok=True)
SCORER_PATH = SCORER_DIR / "suspicion.joblib"
if RUN_SCORER_TRAINING:
    cmd = [sys.executable, str(REPO / "scripts/train_suspicion_scorer.py"),
           "--out", str(SCORER_PATH), "--targets", "4",
           "--attacks", "entity_swap,negation,instruction_injection,sybil,adaptive"]
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0: raise RuntimeError("scorer training failed")
    assert SCORER_PATH.exists()
    SCORER_OVERRIDE = f"scorer.model_path={SCORER_PATH}"
    print("trained:", SCORER_PATH, "report:", SCORER_PATH.with_suffix(".report.json"))
else:
    SCORER_OVERRIDE = None
    print("scorer training skipped")

## 14. Attack-by-baseline matrix

Runs the same corpus, query stream, attack schedule and labels through every
policy. Outputs include `results.json`, `results.csv`, per-query streams,
measured answer ASR and explicit missing-data markers. `RUN_MATRIX=False` is the
safe default; the full command is in `docs/RESEARCH_PROTOCOL.md`.

In [ ]:
RUN_MATRIX = False
MATRIX_DIR = ACTIVE_RUNS / "pilot_matrix"
if RUN_MATRIX:
    cmd = [sys.executable, str(REPO / "scripts/run_research_benchmark.py"),
           "--corpus", str(PILOT_CORPUS), "--queries", str(PILOT_QUERIES),
           "--qrels", str(PILOT_QRELS), "--config", str(PILOT_CONFIG),
           "--out", str(MATRIX_DIR), "--answers", str(PILOT_ANSWERS), "--targets", "20",
           "--attacks", "black_box,white_box,sybil,coordinated,adaptive",
           "--baselines", "no_defence,suspicion_only,corroboration_only,trust_no_deferred,always_on_loo,RAGLedger",
           "--seeds", "20260921", "--scope", "PILOT MATRIX - REAL MODELS"]
    for item in PILOT_OVERRIDES + ([SCORER_OVERRIDE] if SCORER_OVERRIDE else []):
        cmd += ["--set", item]
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0: raise RuntimeError("matrix failed")
    print(json.dumps(json.loads((MATRIX_DIR / "results.json").read_text()), indent=2)[:2000])
else:
    print("Matrix skipped (RUN_MATRIX=False).")

## 15. Ablation studies

A0 standard RAG, A1 suspicion-only, A2 trust-only, A3 corroboration-only, A4
trust without deferred verification and A7 full TRACE-RAG are separate policies
over the same stream. Use the same seeds and dataset as the main matrix.

In [ ]:
RUN_ABLATION = False
ABLATION_DIR = ACTIVE_RUNS / "pilot_ablation"
if RUN_ABLATION:
    cmd = [sys.executable, str(REPO / "scripts/run_research_benchmark.py"),
           "--corpus", str(PILOT_CORPUS), "--queries", str(PILOT_QUERIES),
           "--qrels", str(PILOT_QRELS), "--config", str(PILOT_CONFIG),
           "--out", str(ABLATION_DIR), "--targets", "20",
           "--attacks", "black_box,white_box,sybil,coordinated,adaptive",
           "--baselines", "no_defence,suspicion_only,trust_threshold,corroboration_only,trust_no_deferred,RAGLedger",
           "--seeds", "20260921", "--scope", "PILOT ABLATION - REAL MODELS"]
    for item in PILOT_OVERRIDES + ([SCORER_OVERRIDE] if SCORER_OVERRIDE else []):
        cmd += ["--set", item]
    proc = subprocess.run(cmd, cwd=REPO, text=True)
    if proc.returncode != 0: raise RuntimeError("ablation failed")
else:
    print("Ablation skipped (RUN_ABLATION=False).")

## 16. Robustness and multi-seed runs

Independent seeds are separate streams. Report mean, standard deviation and a
bootstrap interval; do not treat repeated queries as independent samples. The
attack generators carry explicit source/family structure and duplication counts.

In [ ]:
RUN_ROBUSTNESS = False
ROBUSTNESS_DIR = ACTIVE_RUNS / "pilot_robustness"
if RUN_ROBUSTNESS:
    cmd = [sys.executable, str(REPO / "scripts/run_research_benchmark.py"),
           "--corpus", str(PILOT_CORPUS), "--queries", str(PILOT_QUERIES),
           "--qrels", str(PILOT_QRELS), "--config", str(PILOT_CONFIG),
           "--out", str(ROBUSTNESS_DIR), "--targets", "20",
           "--attacks", "entity_swap,black_box,white_box,sybil,coordinated,adaptive",
           "--baselines", "no_defence,RAGLedger",
           "--seeds", "20260921,20260922,20260923", "--scope", "PILOT ROBUSTNESS - REAL MODELS"]
    for item in PILOT_OVERRIDES + ([SCORER_OVERRIDE] if SCORER_OVERRIDE else []):
        cmd += ["--set", item]
    if subprocess.run(cmd, cwd=REPO).returncode != 0:
        raise RuntimeError("robustness run failed")
else:
    print("Robustness skipped (RUN_ROBUSTNESS=False).")

## 17. Statistical analysis

Avoids inferential claims from a single seed: bootstrap intervals are emitted
for rates, and the output table keeps answer ASR, exposure, precision/recall,
false positives, p99 latency and memory in separate columns.

In [ ]:
from trace_rag.evaluation.statistics import bootstrap_ci
import csv

def analyse(path, label):
    if not path.exists():
        print("no measured results for", label); return None
    payload = json.loads(path.read_text()); groups = {}
    for r in payload.get("results", []):
        groups.setdefault((r["case"]["attack"], r["case"]["baseline"]), []).append(r)
    rows = []
    for (attack, baseline), items in sorted(groups.items()):
        def mean(key):
            vals = [r[key] for r in items if r.get(key) is not None]
            return sum(vals) / len(vals) if vals else None
        def ci(key):
            vals = [r[key] for r in items if r.get(key) is not None]
            return list(bootstrap_ci(vals, seed=20260921)) if len(vals) > 1 else [None, None]
        rows.append({"attack": attack, "baseline": baseline, "n": len(items),
                     "exposure_mean": mean("poison_exposure_rate"), "exposure_ci": ci("poison_exposure_rate"),
                     "answer_asr_mean": mean("answer_level_asr"), "answer_asr_ci": ci("answer_level_asr"),
                     "precision_mean": mean("precision"), "recall_mean": mean("recall"),
                     "fpr_mean": mean("false_positive_rate"),
                     "p99_ms_mean": mean("p99_latency_ms"),
                     "peak_memory_mb_mean": mean("peak_memory_mb")})
    out = DRIVE / "exports" / label; out.mkdir(parents=True, exist_ok=True)
    (out / "analysis.json").write_text(json.dumps({"scope": payload.get("scope"), "rows": rows}, indent=2))
    with (out / "analysis.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]) if rows else ["attack"])
        w.writeheader(); w.writerows(rows)
    print(json.dumps(rows, indent=2)[:3000])
    return out

ANALYSIS_DIR = analyse(MATRIX_DIR / "results.json", "pilot_matrix_analysis")
if RUN_ABLATION:
    ABLATION_ANALYSIS_DIR = analyse(ABLATION_DIR / "results.json", "pilot_ablation_analysis")

## 18. Paper artifacts

Packages only measured files. A missing matrix, ablation or robustness file stays
explicitly marked `exists: false`; the manuscript must not cite it until the run
exists.

In [ ]:
PAPER = DRIVE / "paper_inputs"; PAPER.mkdir(parents=True, exist_ok=True)
paths = {"pilot_matrix": MATRIX_DIR / "results.json",
         "pilot_analysis": DRIVE / "exports/pilot_matrix_analysis/analysis.json",
         "environment": DRIVE / "logs/environment.json",
         "scorer_report": SCORER_PATH.with_suffix(".report.json"),
         "ablation": ABLATION_DIR / "results.json",
         "robustness": ROBUSTNESS_DIR / "results.json"}
manifest = {"artifacts": {k: {"path": str(v), "exists": v.exists()} for k, v in paths.items()},
            "claims": {"real_model_run": (PILOT_RUN / "metrics.json").exists(),
                       "matrix": (MATRIX_DIR / "results.json").exists(),
                       "ablation": (ABLATION_DIR / "results.json").exists(),
                       "robustness": (ROBUSTNESS_DIR / "results.json").exists(),
                       "answer_asr_measured": (MATRIX_DIR / "results.json").exists(),
                       "rule": "Only present artifacts may be cited; otherwise NOT RUN."}}
for name, path in paths.items():
    if path.exists():
        shutil.copy2(path, PAPER / f"{name}{path.suffix}")
(PAPER / "paper_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## 19. Export and preserve final results

Active databases, the FAISS index and stream files exist only in `/content` and do not
survive a runtime reset. At every completed step the runner writes an atomic snapshot to
`MyDrive/RAGLedger/backups/<run>/`: consistent SQLite backups, checkpoint, manifest, stream
and the prepared index. The exporter uses the completed active run when present and falls
back to the latest completed Drive snapshot after a reset; it records which source was used.
It exports all three SQLite databases, the manifest, metrics, stream, history, checkpoint and
config snapshot. The large prepared index is not duplicated into the archive; its single
persistent copy remains in the backup directory for a future runtime.


In [ ]:
def export_run(path, label, backup):
    summary = snapshot_summary(backup)
    if (path / "metrics.json").exists():
        source, source_kind = path, "active"
    elif summary.get("snapshot_name"):
        # /content is temporary. A runtime reset must not make a completed
        # experiment unexportable: fall back to the final Drive snapshot.
        source = backup / summary["snapshot_name"]
        source_kind = f"snapshot:{summary['snapshot_name']}"
    else:
        raise RuntimeError(f"{path} has no metrics and no completed Drive snapshot")
    if not (source / "metrics.json").exists():
        raise RuntimeError(f"{source_kind} is not a completed run (metrics.json is absent)")
    manifest = json.loads((source / "run_manifest.json").read_text())
    if summary and manifest.get("signature") != summary.get("run_signature"):
        raise RuntimeError("active run signature does not match the Drive snapshot")
    target = DRIVE / "exports" / label
    target.mkdir(parents=True, exist_ok=True)
    for name in ("run_manifest.json", "metrics.json", "stream.jsonl", "trust_history.csv",
                 "checkpoint.json"):
        if (source / name).exists(): shutil.copy2(source / name, target / name)
    sqlite_names = summary.get("sqlite_files") or ["trust.sqlite3"]
    for name in sqlite_names:
        database = source / name
        if not database.exists():
            raise RuntimeError(f"export source is missing SQLite database {name}")
        src = sqlite3.connect(database)
        dst = sqlite3.connect(target / name)
        try:
            src.backup(dst)
        finally:
            dst.close(); src.close()
    (target / "config_snapshot.json").write_text(json.dumps(manifest["config"], indent=2))
    (target / "latest_snapshot.json").write_text(json.dumps(summary, indent=2))
    provenance = {"schema": "ragledger.export.v1", "source": source_kind,
                  "run_signature": manifest.get("signature"),
                  "snapshot_completed_steps": summary.get("completed_steps"),
                  "index_snapshot": {"digest": summary.get("index_digest"),
                                     "files": summary.get("index_files", [])}}
    (target / "export_provenance.json").write_text(json.dumps(provenance, indent=2))
    if (source / "figures").exists():
        shutil.copytree(source / "figures", target / "figures", dirs_exist_ok=True)
    archive = DRIVE / "exports" / f"{label}_{time.strftime('%Y%m%d-%H%M%S')}"
    shutil.make_archive(str(archive), "gzip", root_dir=target)
    print("exported:", source_kind, "->", archive.with_suffix(".tar.gz"))
export_run(PILOT_RUN, "pilot_qwen25_7b_q4", PILOT_BACKUP)
RESEARCH_BACKUP = BACKUPS / "research_llama31_8b_bf16"
if RUN_RESEARCH and ((ACTIVE_RUNS / "research_llama31_8b_bf16/metrics.json").exists()
                     or "metrics.json" in snapshot_summary(RESEARCH_BACKUP).get("files", [])):
    export_run(ACTIVE_RUNS / "research_llama31_8b_bf16", "research_llama31_8b_bf16",
               RESEARCH_BACKUP)


## Validation status and remaining work

Statically validated in this repository update: notebook JSON and all 15 code cells parse;
the notebook still has 30 cells and 13 sections. The functional preflight uses a disposable
storage tree and the notebook asserts that a fresh experiment directory stays untouched.
Executed locally: the complete offline test suite, coverage, ResourceWarning run, compileall,
and an offline mini stream. **Not executed:** this notebook in Google Colab; no GPU, NLI
checkpoint, Ollama model, vLLM server or NQ download was available locally. No Colab
success is claimed.

Remaining before genuine research: run this notebook on a Colab GPU and verify the pinned
dependency stack; complete the pilot with an exact 5,000/20 subset and a verified immutable
Ollama digest; then use an adequate GPU, prepare exactly 200,000/500 with 100% gold coverage,
start pinned Llama, and run the separate research cell. The full attack × baseline benchmark,
ASR criteria, baselines and confidence intervals remain Person C work in `docs/PERSON_C_TODO.md`.
